# EDA — StatsBomb World Cup 2022

**How this notebook works:** sections marked `WORKED` are complete — run them, read them, understand every line. Sections marked `TODO` are yours. Hints are in comments; solutions are *not* provided anywhere. If you get stuck for more than 20 minutes, ask — but try first.

**The question driving everything:** *can we trust this data enough to build a prediction model and a player-valuation metric on it?* Every cell serves that question.

In [ ]:
import sys
sys.path.insert(0, "..")   # make project modules importable

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from statsbombpy import sb

pd.set_option("display.max_columns", 50)

# One tournament, all matches. ~1 small download.
matches = sb.matches(competition_id=43, season_id=106)  # WC 2022
print(matches.shape)

## 1 — First look (WORKED)

Rule one of EDA: **look at raw rows before any aggregate.** You are checking: do the dtypes make sense? Are there nulls where there shouldn't be? Does one row mean what I think it means?

In [ ]:
# The three commands you run on EVERY new dataset, in this order:
display(matches.head(3))                  # what does a row look like?
display(matches.dtypes.to_frame("dtype")) # are types sane? (dates as dates, scores as ints)
display(
    matches.isna().sum()[lambda s: s > 0].to_frame("n_missing")  # where are the holes?
)

**Answer in this cell (double-click to edit):**
1. What does one row represent?
2. Which columns have missing values — and is each one *expected* missing (e.g. no penalty shootout happened) or *suspicious* missing?
3. `competition_stage` is a plain string. The original loader code assumed it was a dict and silently produced an empty training set. What single command would have caught that on day one?

## 2 — One variable: how are goals distributed? (WORKED, then TODO)

Goals are rare events in a fixed window → the canonical model is **Poisson**. If goals-per-match is roughly Poisson, the variance ≈ the mean, and that single fact explains why football is so upset-prone — and why your prediction model should output probabilities, never certainties.

Finance bridge: this is the same reasoning as modeling rare-event arrivals (defaults, large trades). Distribution first, point estimates second.

In [ ]:
from scipy import stats as scistats  # if missing: %pip install scipy

total_goals = matches["home_score"] + matches["away_score"]
lam = total_goals.mean()
print(f"mean goals/match = {lam:.2f}, variance = {total_goals.var():.2f}  (Poisson => roughly equal)")

observed = total_goals.value_counts(normalize=True).sort_index()
ks = np.arange(0, observed.index.max() + 1)
poisson_pmf = scistats.poisson.pmf(ks, lam)

fig = go.Figure()
fig.add_bar(x=observed.index, y=observed.values, name="observed")
fig.add_scatter(x=ks, y=poisson_pmf, mode="lines+markers", name=f"Poisson(\u03bb={lam:.2f})")
fig.update_layout(title="Goals per match: observed vs Poisson", xaxis_title="goals", yaxis_title="P")
fig.show()

In [ ]:
# TODO: repeat the comparison for HOME goals and AWAY goals separately.
# Q: is there a home-advantage gap at a World Cup (neutral venues)?
# Hint: two bars per goal count, or two overlaid Poisson fits.
# Q: which 2022 'home' teams weren't actually neutral? (Qatar...)


## 3 — Two variables: does xG actually predict goals? (TODO)

This needs event data. The loader's `get_team_match_stats` builds one row per team per match — reuse it rather than re-deriving (it now filters shootout events; see section 4a for why).

⚠️ First run downloads 64 match event files (~3–5 min). Subsequent calls in the same session are cached.

In [ ]:
from data.ingest.statsbomb_loader import StatsBombLoader

team_stats = StatsBombLoader.get_team_match_stats(competition_id=43, season_id=106)
print(team_stats.shape)

# TODO 3a: scatter xg (x) vs goals_for (y), one point per team-match.
#          Add the y=x diagonal. Points above = finished above expectation.
# TODO 3b: compute the Pearson correlation. Then compute it again on
#          TEAM TOURNAMENT TOTALS (groupby team, sum). Which is higher? Why?
#          (Keywords to look up: aggregation reduces variance, law of large numbers)
# TODO 3c: who are the 3 biggest over- and under-performers vs xG?
#          You should find Argentina among the underperformers in the group
#          stage. They won the cup. What does that tell you about n=3 samples?


## 4 — Data-quality hunt: find the bugs this repo shipped with

Real datasets lie to you. This codebase originally had two data bugs that survived until someone *looked*. Re-discover both from raw data — this section is the most interview-valuable thing in the notebook.

In [ ]:
# 4a (GUIDED) — The 5.9 xG final.
# The 2022 final (Argentina 3-3 France, won on penalties) showed xG of 5.89
# for Argentina when summing shot_statsbomb_xg naively. Actual open-play xG
# was ~2.7. Find the discrepancy yourself:

events = StatsBombLoader.get_events_for_match(3869685)  # the final
shots = events[events["type"] == "Shot"]

# TODO: group shots by the `period` column: count and xG sum per period.
# What is period 5? What's the average xG of a shot there, and why ~0.78?
# Q: which OTHER aggregate columns does this contaminate? (think: goals, shots,
#    a striker's goals_p90 if they took a shootout pen)


In [ ]:
# 4b (GUIDED) — The metric that isn't what it says.
# data/ingest/statsbomb_loader.py line ~140 computes a column called
# `pass_accuracy` as:   key_passes / passes * 100
# TODO: explain why that is NOT pass accuracy. Then compute REAL pass
# completion for this match's players:
#   - in StatsBomb events, a Pass row with NaN `pass_outcome` = COMPLETED
#   - completion = completed / attempted
# Sanity-check: elite midfielders complete ~85-93%. If you get 8%, you've
# recomputed the bug. (This mislabel is still in the code — fixing it is
# your first real PR to this repo.)


## 5 — Group comparisons (TODO)

Stage-to-stage and team-to-team differences. Watch your n.

In [ ]:
# TODO 5a: goals per match by stage (group vs knockout). Knockouts are
#          famously tighter — is that visible? How many knockout matches is
#          your conclusion resting on? State n in the chart title.
# TODO 5b: top 8 teams by avg pressures per match (a pressing-intensity proxy).
#          Then check: does pressing correlate with winning, or with WHO they
#          played? (Hint: weaker teams defend more, get pressed less.)


## 6 — Why per-90? Playing-time bias (WORKED skeleton + TODO)

The player-valuation model (CPCS) rests entirely on per-90 normalization. Prove to yourself it matters — and find where it *breaks*.

In [ ]:
players = StatsBombLoader.get_player_tournament_stats(competition_id=43, season_id=106, min_minutes=45)

# TODO 6a: rank top-10 by raw `goals`, then by `goals_p90`. Who enters the
#          per-90 list that wasn't in the raw list? Look up their minutes.
# TODO 6b: scatter minutes_played (x) vs goals_p90 (y). Notice the variance
#          explosion at low minutes — a 60-minute player with 2 goals shows
#          3.0 g/90. This is small-sample noise, not skill.
#          THE per-90 caveat: rates stabilize only with volume.
# TODO 6c: propose and implement ONE guard (min-minutes floor? shrinkage
#          toward the mean?). Finance bridge: same problem as annualizing
#          the return of a 2-week-old fund.


## 7 — Write-up (TODO — the deliverable)

Five findings, each in the format **finding → so what → recommendation**, written for a non-technical reader. No jargon, no chart references. Two must come from the data-quality section.

---

**Finding 1:** ...

**Finding 2:** ...

**Finding 3:** ...

**Finding 4:** ...

**Finding 5:** ...